In [ ]:
!pip install langgraph langchain-openai python-dotenv

In [ ]:
import os
from google.colab import userdata

# Retrieve the secret key from Colab
openai_key = userdata.get('OPENAI_API_KEY')

# Set it as an environment variable for OpenAI/LangChain libraries
os.environ["OPENAI_API_KEY"] = openai_key

print("OpenAI API key loaded successfully!")

In [ ]:
from typing import Annotated
import operator
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class TicketState(TypedDict):
    customer_message: str
    log: Annotated[list, operator.add]

def log_received(state: TicketState) -> dict:
    return {"log": [f"Received: {state['customer_message']}"]}

def log_assigned(state: TicketState) -> dict:
    return {"log": ["Assigned to support queue"]}

builder = StateGraph(TicketState)
builder.add_node("log_received", log_received)
builder.add_node("log_assigned", log_assigned)
builder.add_edge(START, "log_received")
builder.add_edge("log_received", "log_assigned")
builder.add_edge("log_assigned", END)
graph = builder.compile()

result = graph.invoke({"customer_message": "My invoice looks wrong", "log": []})
print(result)

In [3]:
from langgraph.graph import MessagesState

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini")

def run_model(state: MessagesState) -> dict:
    system = SystemMessage("You are a support agent for a SaaS product. "
                           "Be concise and helpful.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

builder = StateGraph(MessagesState)
builder.add_node("run_model", run_model)
builder.add_edge(START, "run_model")
builder.add_edge("run_model", END)

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("My dashboard isn't loading. What should I try?")]})
print(result["messages"][-1].content)

Here are some steps you can try to resolve the issue:

1. **Refresh the Page**: Sometimes a simple refresh can fix loading issues.
2. **Clear Browser Cache**: Clear your browser’s cache and cookies, then try reloading the dashboard.
3. **Check Internet Connection**: Ensure you have a stable internet connection.
4. **Try a Different Browser**: Open the dashboard in another browser to see if the issue persists.
5. **Disable Browser Extensions**: Sometimes extensions can interfere. Try disabling them temporarily.
6. **Check for Service Outages**: Visit the status page of our service to see if there are any ongoing issues.
7. **Log Out and Log Back In**: This can sometimes resolve session-related issues.

If the problem continues, please reach out to support with details about the issue and any error messages you've encountered.


In [7]:
from langchain_core.tools import tool

@tool
def get_customer_tier(customer_id: str) -> str:
    """Look up the subscription tier for a customer by their ID.
    Returns 'free', 'pro', or 'enterprise'."""
    tiers = {
        "cust_1001": "enterprise",
        "cust_2002": "pro",
        "cust_3003": "free",
    }
    return tiers.get(customer_id, "not found")

In [8]:
tools = [get_customer_tier]
llm_with_tools = llm.bind_tools(tools)

def run_model(state: MessagesState) -> dict:
    system = SystemMessage("You are a support agent for a SaaS product. "
                           "Use available tools when you need account-specific information.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [9]:
from langgraph.prebuilt import ToolNode, tools_condition

tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("run_model", run_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "run_model")
builder.add_conditional_edges("run_model", tools_condition)
builder.add_edge("tools", "run_model")

graph = builder.compile()

In [10]:
result = graph.invoke({"messages": [
    HumanMessage("Can you check what plan customer cust_1001 is on?")
]})

for msg in result["messages"]:
    print(type(msg).__name__, ":", msg.content or msg.tool_calls)

HumanMessage : Can you check what plan customer cust_1001 is on?
AIMessage : [{'name': 'get_customer_tier', 'args': {'customer_id': 'cust_1001'}, 'id': 'call_5Rnc4pF6AJx53oiCfhAZYKDu', 'type': 'tool_call'}]
ToolMessage : enterprise
AIMessage : Customer cust_1001 is on the enterprise plan.


In [12]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [13]:
config = {"configurable": {"thread_id": "ticket-7741"}}

graph.invoke(
    {"messages": [HumanMessage("Hi, I can't access my account.")]},
    config,
)

result = graph.invoke(
    {"messages": [HumanMessage("My ID is cust_2002, can you check my plan?")]},
    config,
)

print(result["messages"][-1].content)

Your current subscription plan is the "Pro" tier. If you're still having difficulty accessing your account, please let me know what error message you're encountering or any other details, and I'll assist you further!
